# Project 01 -- Portfolio Risk Dashboard: Walkthrough

**Author:** Andres Gonzalez Ortega  
**Objective:** Compute, compare, and backtest Value-at-Risk (VaR) and Expected Shortfall (ES) for a multi-asset portfolio using historical simulation, parametric (normal), and Monte Carlo methods.

**Portfolio:** SPY, QQQ, IWM, TLT, GLD (equal-weighted)

---

## Table of Contents

1. Setup and configuration
2. Data loading and return computation
3. VaR estimation (three methods) and ES
4. Backtesting (Kupiec, Christoffersen, traffic-light)
5. Visualizations

In [ ]:
import warnings

warnings.filterwarnings("ignore")

# Project-level imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Shared library imports
from risk_analyst.data.market import compute_losses, compute_returns, fetch_prices
from risk_analyst.measures.backtesting import backtest_var
from risk_analyst.visualization.risk_plots import (
    plot_correlation_heatmap,
    plot_loss_distribution,
    plot_rolling_volatility,
    plot_var_backtest,
)

_proj_src = str(Path("..").resolve() / "src")
if _proj_src not in sys.path:
    sys.path.insert(0, _proj_src)
from model import RiskModel

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

SEED = 42
print("Setup complete.")

## 1. Configuration

All parameters are externalized in `configs/default.yaml`. We define them here
for transparency and easy modification within the notebook.

In [ ]:
# Portfolio definition
TICKERS = ["SPY", "QQQ", "IWM", "TLT", "GLD"]
START_DATE = "2015-01-01"
END_DATE = None  # latest available

# Equal weights
N_ASSETS = len(TICKERS)
WEIGHTS = np.ones(N_ASSETS) / N_ASSETS

# Risk parameters
CONFIDENCE_LEVELS = [0.95, 0.99]
ROLLING_WINDOW = 252  # 1 trading year
MC_SIMS = 10_000

print(f"Portfolio: {TICKERS}")
print(f"Weights:   {WEIGHTS}")
print(f"Period:    {START_DATE} to present")

## 2. Data Loading and Return Computation

We fetch adjusted close prices from Yahoo Finance and compute log returns.
Portfolio losses are defined as $L_t = -r_{p,t}$ (positive = adverse move).

In [ ]:
# Fetch prices
prices = fetch_prices(TICKERS, start_date=START_DATE, end_date=END_DATE)
print(f"Prices shape: {prices.shape}")
print(f"Date range:   {prices.index[0].date()} to {prices.index[-1].date()}")
prices.tail()

In [ ]:
# Compute log returns
returns = compute_returns(prices, method="log")
print(f"Returns shape: {returns.shape}")
returns.describe()

In [ ]:
# Compute portfolio losses
losses = compute_losses(returns, WEIGHTS)
print(f"Losses shape: {losses.shape}")
print(f"Mean loss:    {losses.mean():.6f}")
print(f"Std loss:     {losses.std():.6f}")
print(f"Max loss:     {losses.max():.6f} (worst day)")
print(f"Min loss:     {losses.min():.6f} (best day, negative = gain)")

## 3. VaR Estimation -- Three Methods

We compute Value-at-Risk using:
1. **Historical simulation** -- empirical quantile: $\text{VaR}_\alpha = F_L^{-1}(\alpha)$
2. **Parametric (normal)** -- $\text{VaR}_\alpha = \mu_L + \sigma_L \cdot \Phi^{-1}(\alpha)$
3. **Monte Carlo** -- bootstrap resampling from the observed loss distribution

And **Expected Shortfall** (CVaR): $\text{ES}_\alpha = E[L \mid L \geq \text{VaR}_\alpha]$

In [ ]:
# Fit the RiskModel
model = RiskModel(n_sims=MC_SIMS, seed=SEED)
model.fit(returns, WEIGHTS)

# Compute VaR and ES for all method x alpha combinations
results = []
for alpha in CONFIDENCE_LEVELS:
    for method in ["historical", "parametric", "monte_carlo"]:
        var_val = model.var(alpha=alpha, method=method)
        es_val = model.es(alpha=alpha)
        results.append({
            "alpha": alpha,
            "method": method,
            "VaR": var_val,
            "ES": es_val,
        })

summary = pd.DataFrame(results)
print("Risk Measures Summary")
print("=" * 55)
summary.style.format({"VaR": "{:.6f}", "ES": "{:.6f}", "alpha": "{:.0%}"})

### Loss Distribution with VaR and ES

The histogram below shows the empirical loss distribution. VaR marks the threshold
that is exceeded with probability $1-\alpha$; ES is the average loss in the tail beyond VaR.

In [ ]:
# Loss distribution at 99% confidence
var_99 = model.var(alpha=0.99, method="historical")
es_99 = model.es(alpha=0.99)

fig = plot_loss_distribution(
    losses.values, var=var_99, es=es_99,
    title="Portfolio Loss Distribution (99% VaR and ES)"
)
plt.show()

## 4. Backtesting

We compute rolling 252-day VaR and evaluate its accuracy using three standard tests:

- **Kupiec (1995):** Tests whether the observed violation rate matches the expected rate (unconditional coverage). $H_0$: violation rate $= 1 - \alpha$.
- **Christoffersen (1998):** Tests whether violations are independent (no clustering). $H_0$: violations are i.i.d.
- **Basel traffic-light:** Classifies the model as green (0--4 violations / 250 days), yellow (5--9), or red (10+).

In [ ]:
# Compute rolling VaR for each method at 99%
alpha = 0.99
methods = ["historical", "parametric", "monte_carlo"]
rolling_vars = {}

for method in methods:
    rolling_vars[method] = model.rolling_var(
        window=ROLLING_WINDOW, alpha=alpha, method=method
    )

# Run backtests
print(f"Backtesting Results (alpha = {alpha})")
print("=" * 80)

backtest_rows = []
for method in methods:
    rv = rolling_vars[method]
    valid = ~rv.isna()
    losses_valid = model.losses[valid.values]
    var_valid = rv[valid].values

    report = backtest_var(losses=losses_valid, var_series=var_valid, alpha=alpha)

    backtest_rows.append({
        "Method": method,
        "Violations": report.n_violations,
        "N_obs": report.n_obs,
        "Viol. Rate": f"{report.violation_rate:.3%}",
        "Expected": f"{report.expected_rate:.3%}",
        "Kupiec LR": f"{report.kupiec.statistic:.3f}",
        "Kupiec p": f"{report.kupiec.p_value:.4f}",
        "Kupiec Reject": report.kupiec.reject,
        "Chr. LR": f"{report.christoffersen.statistic:.3f}",
        "Chr. p": f"{report.christoffersen.p_value:.4f}",
        "Traffic Light": report.traffic_light.zone,
    })

bt_df = pd.DataFrame(backtest_rows)
bt_df

## 5. Visualizations

### 5.1 VaR Backtest Plot

The plot shows realized losses (blue) vs. the rolling VaR forecast (red dashed).
Red dots mark **violations** -- days where the loss exceeded VaR.

In [ ]:
# Plot VaR backtest for historical method at 99%
rv_hist = rolling_vars["historical"]
valid = ~rv_hist.isna()

fig = plot_var_backtest(
    losses=losses[valid],
    var_series=rv_hist[valid],
    title="Historical VaR (99%) Backtest",
)
plt.show()

### 5.2 Rolling Volatility

Annualized rolling volatility (21-day window) for each asset in the portfolio.

In [ ]:
fig = plot_rolling_volatility(returns, window=21)
plt.show()

### 5.3 Correlation Heatmap

The return correlation matrix reveals diversification (or lack thereof) among assets.

In [ ]:
fig = plot_correlation_heatmap(returns)
plt.show()

### 5.4 Comparing VaR Methods at 95%

Side-by-side backtest visualization for all three VaR methods at the 95% level.

In [ ]:
# Rolling VaR at 95% for comparison
alpha_95 = 0.95
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for i, method in enumerate(methods):
    rv = model.rolling_var(window=ROLLING_WINDOW, alpha=alpha_95, method=method)
    valid = ~rv.isna()

    losses_v = losses[valid].values
    var_v = rv[valid].values
    x = losses[valid].index
    violations = losses_v >= var_v

    axes[i].plot(x, losses_v, color="steelblue", linewidth=0.6, alpha=0.8, label="Loss")
    axes[i].plot(x, var_v, color="firebrick", linewidth=1.0, linestyle="--", label=f"VaR ({method})")
    viol_idx = np.where(violations)[0]
    axes[i].scatter(x[viol_idx], losses_v[viol_idx], color="red", s=12, zorder=5)
    axes[i].set_title(f"{method.replace('_', ' ').title()} VaR (95%)", fontsize=11)
    axes[i].legend(loc="upper left", fontsize=9)
    axes[i].grid(alpha=0.3)

fig.suptitle("VaR Method Comparison -- 95% Confidence", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## Conclusion

This notebook demonstrated the full pipeline for portfolio risk analysis:

1. **Data**: Downloaded daily adjusted prices for a 5-asset portfolio from Yahoo Finance.
2. **VaR**: Computed Value-at-Risk via historical simulation, parametric (normal), and Monte Carlo bootstrap methods at 95% and 99% confidence.
3. **ES**: Computed Expected Shortfall as the average loss in the tail -- a coherent risk measure (Acerbi & Tasche, 2002).
4. **Backtesting**: Evaluated VaR accuracy using Kupiec unconditional coverage, Christoffersen conditional coverage, and Basel traffic-light tests.
5. **Visualization**: Produced backtest overlays, rolling volatility, correlation heatmaps, and loss distributions.

Key observations:
- **Parametric VaR** tends to underestimate tail risk because real return distributions have fatter tails than the normal.
- **Historical simulation** is non-parametric but sensitive to the lookback window.
- **Monte Carlo** (bootstrap) closely mirrors historical VaR when resampling from the same data.
- **ES** is always larger than VaR, capturing the severity of tail events beyond the VaR threshold.

Next steps: Project 02 (Monte Carlo Engine) extends this with variance reduction techniques and full revaluation.